In [ ]:
from google.colab import drive
drive.mount('/content/drive')


from google.colab import drive
import os
import sys


# 2. Define your paths
base_path = "/content/drive/MyDrive/Environemnt"
depth_dir = os.path.join(base_path, "depthAnything")
mask_dir = os.path.join(base_path, "mask2former_model")


# 3. Install required libraries (Must be done every session)
!pip install -q transformers timm accelerate scikit-learn

import torch
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

# import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
# from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression

# Now import

sys.path.append(os.path.join(depth_dir, 'Depth-Anything-V2'))
from depth_anything_v2.dpt import DepthAnythingV2


device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. LOAD DEPTH ANYTHING V2 ---
print("Loading Depth-Anything-V2 from Drive...")
depth_weights = os.path.join(depth_dir, 'depth_anything_v2_vits.pth')
depth_model = DepthAnythingV2(encoder='vits', features=64, out_channels=[48, 96, 192, 384])
depth_model.load_state_dict(torch.load(depth_weights, map_location=device, weights_only=False))
depth_model.to(device).eval()


# --- 3. LOAD MASK2FORMER (Permanent Storage) ---
print("Loading Mask2Former...")
model_name = "facebook/mask2former-swin-tiny-ade-semantic"

m2f_processor = AutoImageProcessor.from_pretrained(model_name, cache_dir=mask_dir)

# FIX: Add .to(device) here
m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    model_name,
    cache_dir=mask_dir
).to(device).eval()



# -*- coding: utf-8 -*-
"""
Room Tiling Visualizer v21
- Fixes: "Dirty Walls" effect (Lighting map now ignores cracks/marks).
- Tech: Replaced CLAHE (Contrast Enhancer) with Bilateral+Gaussian Smoothing.
- Result: Pure ambient lighting shadows without preserving wall defects.
"""

import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression, RANSACRegressor
import torch

# ─────────────────────────────────────────────────────────────
# UTILITY
# ─────────────────────────────────────────────────────────────

def cleanup_small_regions(labels, wall_mask, min_ratio=0.01):
    h, w = labels.shape
    min_size = int(h * w * min_ratio)
    cleaned = labels.copy().astype(np.uint8)
    for val in np.unique(labels):
        if val == 0:
            continue
        mask = np.uint8(labels == val)
        num_labels, comp_labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
        for j in range(1, num_labels):
            if stats[j, cv2.CC_STAT_AREA] < min_size:
                cleaned[comp_labels == j] = 0
    return cleaned


def make_tile_unit(tile_img, tile_w_px, tile_h_px, grout_px):
    grout_color = [235, 233, 228]
    unit = np.full((tile_h_px + grout_px, tile_w_px + grout_px, 3),
                   grout_color, dtype=np.uint8)

    # ── INTELLIGENT RESIZING ────────────────────────────────
    # If the source tile is huge, simple cubic interpolation causes aliasing/shimmering.
    # Use AREA interpolation for high-quality downscaling.
    if tile_img.shape[0] > tile_h_px * 1.5 or tile_img.shape[1] > tile_w_px * 1.5:
        interp = cv2.INTER_AREA
    else:
        interp = cv2.INTER_LANCZOS4

    t = cv2.resize(tile_img, (tile_w_px, tile_h_px), interpolation=interp)
    g2 = grout_px // 2
    unit[g2:g2+tile_h_px, g2:g2+tile_w_px] = t
    return unit


# ─────────────────────────────────────────────────────────────
# WALL LINE DETECTION
# ─────────────────────────────────────────────────────────────

def find_true_wall_lines(plane_mask, depth, img_h, img_w):
    h, w = img_h, img_w

    # Heavy morphological close to fill window/furniture holes
    k_big = np.ones((25, 25), np.uint8)
    filled = cv2.morphologyEx(plane_mask, cv2.MORPH_CLOSE, k_big)
    filled = cv2.dilate(filled, np.ones((5, 5), np.uint8))
    filled = np.clip(filled, 0, 1).astype(np.uint8)

    cols, top_ys, bot_ys = [], [], []
    for x in range(w):
        col_px = np.where(filled[:, x] > 0)[0]
        if len(col_px) < 3:
            continue
        cols.append(x)
        top_ys.append(float(col_px.min()))
        bot_ys.append(float(col_px.max()))

    if len(cols) < 10:
        return None

    cols   = np.array(cols,   dtype=np.float32)
    top_ys = np.array(top_ys, dtype=np.float32)
    bot_ys = np.array(bot_ys, dtype=np.float32)
    X      = cols.reshape(-1, 1)
    x_min  = float(cols[0])
    x_max  = float(cols[-1])

    class _Reg:
        def __init__(self, m, b):
            self.coef_ = [m]; self.intercept_ = b
        def predict(self, X):
            return np.array(X).flatten() * self.coef_[0] + self.intercept_

    # Ceiling line
    try:
        r = RANSACRegressor(residual_threshold=h * 0.015, random_state=0, max_trials=500)
        r.fit(X, top_ys)
        top_m, top_b = float(r.estimator_.coef_[0]), float(r.estimator_.intercept_)
    except Exception:
        r = LinearRegression().fit(X, top_ys)
        top_m, top_b = float(r.coef_[0]), float(r.intercept_)
    reg_top = _Reg(top_m, top_b)

    # Bottom line
    try:
        rb = RANSACRegressor(residual_threshold=h * 0.02, random_state=0, max_trials=500)
        rb.fit(X, bot_ys)
        bot_m_raw = float(rb.estimator_.coef_[0])
        bot_b_raw = float(rb.estimator_.intercept_)
    except Exception:
        rb = LinearRegression().fit(X, bot_ys)
        bot_m_raw, bot_b_raw = float(rb.coef_[0]), float(rb.intercept_)
    reg_bot_raw = _Reg(bot_m_raw, bot_b_raw)

    # Bottom blocked check
    max_visible_bot = float(bot_ys.max())
    expected_floor  = h * 0.65

    # Check depth consistency or blockage
    bottom_blocked = False

    # 1. Floor is way too high?
    if max_visible_bot < expected_floor:
        bottom_blocked = True

    # 2. Parallelism check (Relaxed for side walls)
    slope_diff = abs(bot_m_raw - top_m)
    are_converging = (bot_m_raw * top_m) < 0 # Signs opposite usually means perspective

    if slope_diff > 4.0 and not are_converging:
         bottom_blocked = True

    # 3. Noisy bottom edge?
    elif np.std(bot_ys - reg_bot_raw.predict(X)) > h * 0.08:
        bottom_blocked = True

    if bottom_blocked:
        # Fallback to VP or Parallel
        horizon_y = h * 0.40
        if abs(top_m) > 0.05:
            vp_x = (horizon_y - top_b) / top_m
            vp_x = np.clip(vp_x, -w*2, 3*w)
            x_center = (x_min + x_max) / 2.0
            floor_center_y = h * 0.88
            if abs(x_center - vp_x) > 1.0:
                floor_m = (floor_center_y - horizon_y) / (x_center - vp_x)
                floor_b = horizon_y - floor_m * vp_x
            else:
                floor_m = top_m
                floor_b = floor_center_y - floor_m * x_center
        else:
            floor_m = top_m
            floor_center_y = h * 0.88
            floor_b = floor_center_y - floor_m * ((x_min + x_max)/2.0)
        reg_bot = _Reg(floor_m, floor_b)
    else:
        reg_bot = reg_bot_raw

    x_mid      = np.array([[float(np.median(cols))]])
    wall_h_ref = max(float(reg_bot.predict(x_mid)[0]) - float(reg_top.predict(x_mid)[0]), 1.0)

    return reg_top, reg_bot, x_min, x_max, wall_h_ref


# ─────────────────────────────────────────────────────────────
# TILE CANVAS (RUNNING BOND)
# ─────────────────────────────────────────────────────────────

def build_flat_tile_canvas(tile_img, tile_w_px, tile_h_px, grout_px, n_tiles_w, n_tiles_h):
    grout_color = [235, 233, 228]
    unit_w   = tile_w_px + grout_px
    unit_h   = tile_h_px + grout_px
    canvas_w = n_tiles_w * unit_w + max(grout_px, 0)
    canvas_h = n_tiles_h * unit_h + max(grout_px, 0)
    canvas = np.full((canvas_h, canvas_w, 3), grout_color, dtype=np.uint8)
    tile_r = cv2.resize(tile_img, (tile_w_px, tile_h_px), interpolation=cv2.INTER_LANCZOS4)

    for r in range(n_tiles_h):
        # RUNNING BOND: Shift every odd row by half a tile width
        row_offset = 0
        if r % 2 == 1:
            row_offset = unit_w // 2

        for c in range(-1, n_tiles_w + 1):
            y = r * unit_h + grout_px
            x = c * unit_w + grout_px + row_offset

            # Bounds check
            if x >= canvas_w or y >= canvas_h: continue
            if x + tile_w_px <= 0 or y + tile_h_px <= 0: continue

            ds_x, de_x = max(0, x), min(canvas_w, x + tile_w_px)
            ds_y, de_y = max(0, y), min(canvas_h, y + tile_h_px)

            ss_x, se_x = ds_x - x, ds_x - x + (de_x - ds_x)
            ss_y, se_y = ds_y - y, ds_y - y + (de_y - ds_y)

            if de_x > ds_x and de_y > ds_y:
                canvas[ds_y:de_y, ds_x:de_x] = tile_r[ss_y:se_y, ss_x:se_x]

    return canvas, canvas_w, canvas_h


# ─────────────────────────────────────────────────────────────
# CORE TILING
# ─────────────────────────────────────────────────────────────

def tile_plane_vp(room_img, plane_mask, depth, tile_unit,
                  unit_w, unit_h, lighting, specular,
                  global_ceil_y, global_floor_y,
                  tile_img_raw, tile_w_px, tile_h_px, grout_px):
    h, w = room_img.shape[:2]

    lines = find_true_wall_lines(plane_mask, depth, h, w)
    if lines is None:
        return None, None, plane_mask

    reg_top, reg_bot, x_min, x_max, wall_h_ref = lines

    # Calculate detected corner heights FIRST
    TL_y = reg_top.predict([[x_min]])[0]
    TR_y = reg_top.predict([[x_max]])[0]
    BL_y = reg_bot.predict([[x_min]])[0]
    BR_y = reg_bot.predict([[x_max]])[0]

    h_left  = max(BL_y - TL_y, 1.0)
    h_right = max(BR_y - TR_y, 1.0)

    # ────────────────────────────────────────────────────────
    # DEPTH-GUIDED PERSPECTIVE CORRECTION
    # Use the ACTUAL depth map to determine how much the wall recedes.
    # Depth Map Convention: High Value = Near (Disparity)
    # ────────────────────────────────────────────────────────

    # Get median depth at left and right edges of the mask
    ys, xs = np.where(plane_mask > 0)
    if len(xs) > 0:
        x_min_val = xs.min()
        x_max_val = xs.max()
        x_span = x_max_val - x_min_val

        # Indices in the coordinate arrays
        is_left  = xs < (x_min_val + x_span * 0.1)
        is_right = xs > (x_max_val - x_span * 0.1)

        # Get depth values for the wall pixels
        wall_depths = depth[ys, xs]

        d_left = np.median(wall_depths[is_left]) if np.any(is_left) else 1.0
        d_right = np.median(wall_depths[is_right]) if np.any(is_right) else 1.0

        # Avoid div zero
        d_left = max(d_left, 0.1)
        d_right = max(d_right, 0.1)

        # EXPECTED RATIO: d_right / d_left
        expected_ratio = d_right / d_left

        # If there is a meaningful disagreement, or we just want to enforce the depth:
        if abs(expected_ratio - 1.0) > 0.05: # If there is actual depth difference

             cy_top = (TL_y + TR_y) / 2.0
             cy_bot = (BL_y + BR_y) / 2.0
             avg_h  = (h_left + h_right) / 2.0

             # PERSPECTIVE MATH FLIP (Fixed v18):
             # We want Height to be proportional to Disparity (Depth Value).
             # Near (High Depth) = Tall (Large Height).
             # Far (Low Depth) = Short (Small Height).
             # h_R / h_L = d_R / d_L = expected_ratio
             # h_R = expected_ratio * h_L

             new_h_left  = (2.0 * avg_h) / (1.0 + expected_ratio)
             new_h_right = expected_ratio * new_h_left

             # Apply new heights centered on the same midpoints
             # We apply 70% of this correction (Depth is usually very accurate)
             blend = 0.7
             final_h_left  = h_left * (1-blend) + new_h_left * blend
             final_h_right = h_right * (1-blend) + new_h_right * blend

             # Re-calculate Top/Bottom Ys using these new heights
             mid_L = (TL_y + BL_y) / 2.0
             mid_R = (TR_y + BR_y) / 2.0

             TL_y = mid_L - final_h_left / 2.0
             BL_y = mid_L + final_h_left / 2.0
             TR_y = mid_R - final_h_right / 2.0
             BR_y = mid_R + final_h_right / 2.0

             # Update the regression objects so the "Coverage Expander" uses new slopes
             top_m_new = (TR_y - TL_y) / (x_max - x_min)
             top_b_new = TL_y - top_m_new * x_min
             reg_top.coef_[0] = top_m_new
             reg_top.intercept_ = top_b_new

             bot_m_new = (BR_y - BL_y) / (x_max - x_min)
             bot_b_new = BL_y - bot_m_new * x_min
             reg_bot.coef_[0] = bot_m_new
             reg_bot.intercept_ = bot_b_new

    # ────────────────────────────────────────────────────────
    # COVERAGE EXPANDER (Crucial fix for "Missed wall area")
    # Now that slopes are corrected for 3D depth, we MUST shift
    # the lines outwards to ensure they encompass every pixel of the mask.
    # ────────────────────────────────────────────────────────

    mys, mxs = np.where(plane_mask > 0)
    if len(mys) > 0:
        pred_top = reg_top.predict(mxs.reshape(-1, 1))
        # Find max pixel extending ABOVE the top line
        shift_up = np.max(pred_top - mys)
        if shift_up > 0:
            reg_top.intercept_ -= (shift_up + 2) # Shift line up

        pred_bot = reg_bot.predict(mxs.reshape(-1, 1))
        # Find max pixel extending BELOW the bottom line
        shift_down = np.max(mys - pred_bot)
        if shift_down > 0:
            reg_bot.intercept_ += (shift_down + 2) # Shift line down

    # Final Coordinates for Homography
    TL_y = reg_top.predict([[x_min]])[0]
    TR_y = reg_top.predict([[x_max]])[0]
    BL_y = reg_bot.predict([[x_min]])[0]
    BR_y = reg_bot.predict([[x_max]])[0]

    TL = np.float32([x_min, TL_y])
    TR = np.float32([x_max, TR_y])
    BL = np.float32([x_min, BL_y])
    BR = np.float32([x_max, BR_y])

    # ────────────────────────────────────────────────────────
    # MASK STRAIGHTENING (Fix for "fluctuations")
    # 1. Fill UP to the line (remove dips)
    # 2. Cut DOWN to the line (remove spikes)
    # ────────────────────────────────────────────────────────
    ys_idx, xs_idx = np.indices((h, w))
    ceil_line = reg_top.predict(np.arange(w).reshape(-1, 1)).astype(np.float32)

    # CUT: Remove anything above the line
    below_ceil = (ys_idx >= ceil_line[xs_idx]).astype(np.uint8)
    refined_mask = cv2.bitwise_and(plane_mask, below_ceil)

    # FILL: Fill any gaps between the mask top and the line
    # (Only for columns that actually have wall content)
    col_has_wall = np.any(refined_mask, axis=0) # [w]

    if np.any(col_has_wall):
        # Top-most pixel in each column currently
        # argmax gives index of first '1'. If none, gives 0 (handled by col_has_wall)
        curr_top_y = np.argmax(refined_mask, axis=0) # [w]

        # The target top is ceil_line (clipped to 0)
        target_top_y = np.clip(ceil_line, 0, h-1).astype(np.int32) # [w]

        # Identify pixels that are in the gap
        # GAP = (Y >= target_top_y) AND (Y < curr_top_y) AND (Column Has Wall)
        gap_region = (ys_idx >= target_top_y[xs_idx]) & \
                     (ys_idx < curr_top_y[xs_idx]) & \
                     col_has_wall[xs_idx]

        refined_mask[gap_region] = 1

    wall_w_px  = x_max - x_min
    wall_h_avg = ( (BL_y - TL_y) + (BR_y - TR_y) ) / 2.0

    n_tiles_w = max(int(round(wall_w_px  / unit_w)) + 2, 1)
    n_tiles_h = max(int(round(wall_h_avg / unit_h)) + 2, 1)

    canvas, cw, ch = build_flat_tile_canvas(
        tile_img_raw, tile_w_px, tile_h_px, grout_px, n_tiles_w, n_tiles_h)

    src_pts = np.float32([[0, 0], [cw, 0], [cw, ch], [0, ch]])
    dst_pts = np.float32([TL, TR, BR, BL])

    if cv2.contourArea(dst_pts) < 50: return None, None, plane_mask

    # ────────────────────────────────────────────────────────
    # SUPER-SAMPLING (SSAA) for "Luxury" Quality
    # Renders at 2x resolution then downscales to fix jagged edges/grout
    # ────────────────────────────────────────────────────────
    ss = 2.0
    dst_pts_ss = dst_pts * ss

    M = cv2.getPerspectiveTransform(src_pts, dst_pts_ss)

    # Render High-Res
    warped_ss = cv2.warpPerspective(canvas, M, (int(w * ss), int(h * ss)),
                                  flags=cv2.INTER_LANCZOS4,
                                  borderMode=cv2.BORDER_REPLICATE)

    # Downscale for Anti-Aliasing (Averaging pixels)
    warped = cv2.resize(warped_ss, (w, h), interpolation=cv2.INTER_AREA)

    ys, xs = np.where(refined_mask > 0)
    if len(ys) == 0: return None, None, plane_mask

    warped_f = warped.astype(np.float64) / 255.0
    spec = specular[ys, xs, 0]
    for ch_i in range(3):
        lc = lighting[ys, xs, ch_i]
        warped_f[ys, xs, ch_i] = np.clip(
            warped_f[ys, xs, ch_i] * (lc * 0.78 + 0.22) + spec * 0.25, 0, 1)

    return warped_f, reg_top, refined_mask


# ─────────────────────────────────────────────────────────────
# POST-PROCESS: straighten ceiling edges
# ─────────────────────────────────────────────────────────────

def straighten_ceiling_edges(final_img, plane_labels, reg_top_dict):
    result = final_img.copy()
    h, w = result.shape[:2]
    xs = np.arange(w, dtype=np.float32)

    for label_id, reg_top in reg_top_dict.items():
        ceil_ys = reg_top.predict(xs.reshape(-1, 1)).astype(np.float32)
        for x in range(w):
            cy = int(round(ceil_ys[x]))
            cy = max(0, min(cy, h - 1))
            for dy in range(-3, 4):
                row = cy + dy
                if 0 <= row < h:
                    if dy < 0 and plane_labels[row, x] == label_id:
                        src_row = max(0, cy - 8)
                        result[row, x] = result[src_row, x]
    return result


# ─────────────────────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────────────────────

def process_room_tiling(room_img, tile_img, tile_w_cm, tile_h_cm, orientation, grout_mm):
    h, w = room_img.shape[:2]
    room_rgb = cv2.cvtColor(room_img, cv2.COLOR_BGR2RGB)

    # ── A. DEPTH ──────────────────────────────────────────────
    with torch.no_grad():
        depth = depth_model.infer_image(room_rgb)

    # ── B. SEGMENTATION ───────────────────────────────────────
    inputs = m2f_processor(images=room_img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = m2f_model(**inputs)
    seg = m2f_processor.post_process_semantic_segmentation(
        outputs, target_sizes=[(h, w)])[0].cpu().numpy()

    wall_mask     = (seg == 0).astype(np.uint8)
    obstacle_mask = (seg != 0).astype(np.uint8)

    # ── C. PLANE CLUSTERING ───────────────────────────────────
    depth_blur = cv2.GaussianBlur(depth.astype(np.float32), (5, 5), 0)
    dz_dx  = cv2.Scharr(depth_blur, cv2.CV_32F, 1, 0)
    dz_dy  = cv2.Scharr(depth_blur, cv2.CV_32F, 0, 1)
    normals = np.dstack((-dz_dx, -dz_dy, np.ones_like(depth_blur)))
    normals /= (np.linalg.norm(normals, axis=2, keepdims=True) + 1e-6)

    wall_indices = np.where(wall_mask > 0)
    if len(wall_indices[0]) < 100:
        return room_img, wall_mask

    yy, xx = np.mgrid[0:h, 0:w]
    features = np.hstack((
        normals[wall_indices] * 2.0,
        (xx[wall_indices] / w).reshape(-1, 1) * 0.5,
        (yy[wall_indices] / h).reshape(-1, 1) * 0.5
    ))

    n_samp = min(10000, len(features))
    sidx   = np.random.choice(len(features), n_samp, replace=False)
    kmeans = KMeans(n_clusters=3, n_init=5, random_state=42).fit(features[sidx])

    raw_labels   = np.zeros((h, w), dtype=np.int32)
    raw_labels[wall_indices] = kmeans.predict(features) + 1
    raw_u8       = cv2.medianBlur(raw_labels.astype(np.uint8), 15)
    plane_labels = cleanup_small_regions(raw_u8, wall_mask, min_ratio=0.01)

    # ── D. LIGHTING ───────────────────────────────────────────
    # v21 Change: Remove wall defects (cracks/marks) from lighting map
    gray  = cv2.cvtColor(room_img, cv2.COLOR_BGR2GRAY)

    # 1. Bilateral Filter: Smooths surface texture while keeping strong edges (corners)
    #    d=-1 (auto), sigmaColor=80, sigmaSpace=80 -> Strong smoothing
    smooth = cv2.bilateralFilter(gray, 25, 80, 80)

    # 2. Gaussian Blur: Soften into a general ambient light map
    smooth = cv2.GaussianBlur(smooth, (41, 41), 0)

    lb = smooth.astype(np.float64) / 255.0

    # Slight contrast boost to ensure wall doesn't look washed out
    lb = np.clip(lb * 1.05, 0, 1)

    lighting = cv2.merge([lb] * 3)
    specular = cv2.merge([np.power(lb, 3.5) * 0.55] * 3)

    # ── E. TILE SIZE ──────────────────────────────────────────
    ppm       = h / 2.7
    tile_w_px = max(int((tile_w_cm / 100.0) * ppm), 8)
    tile_h_px = max(int((tile_h_cm / 100.0) * ppm), 8)

    if (orientation == 'v' and tile_w_px > tile_h_px) or \
       (orientation == 'h' and tile_h_px > tile_w_px):
        tile_w_px, tile_h_px = tile_h_px, tile_w_px

    # Grout: convert mm to pixels. h=2.7m. 1mm = 0.001m.
    grout_px = int(round((grout_mm / 1000.0) * ppm))
    if grout_mm > 0 and grout_px < 1:
        grout_px = 1 # Force at least 1 pixel if user requested grout

    # Ensure tiles are always visible — minimum 20px on small images
    min_tile_px = max(20, int(min(h, w) * 0.05))
    tile_w_px   = max(tile_w_px, min_tile_px)
    tile_h_px   = max(tile_h_px, min_tile_px)
    unit_w      = tile_w_px + grout_px
    unit_h      = tile_h_px + grout_px
    tile_unit   = make_tile_unit(tile_img, tile_w_px, tile_h_px, grout_px)

    has_blue_plane = False
    for lid in np.unique(plane_labels):
        if lid == 0: continue

        # Check if the plane touches the ceiling.
        # Blue plane usually touching ceiling, red might be a side wall
        mask_lid = (plane_labels == lid).astype(np.uint8)
        y_coords, _ = np.where(mask_lid > 0)

        # If it touches the top 10% of the image, we force it to tile
        if len(y_coords) > 0 and np.min(y_coords) < h * 0.1:
            has_blue_plane = True

    n_planes = len(np.unique(plane_labels)) - 1
    print(f"  Tile: {tile_w_px}×{tile_h_px} px  |  Grout: {grout_px} px  |  Planes: {n_planes}")

    # ── F. GLOBAL CEILING/FLOOR Y ──────────────────────────────────
    all_wall_ys = np.where(wall_mask > 0)[0]
    if len(all_wall_ys) > 0:
        global_ceil_y  = float(np.percentile(all_wall_ys, 5))
        global_floor_y = float(np.percentile(all_wall_ys, 95))
    else:
        global_ceil_y  = 0.0
        global_floor_y = float(h)
    print(f"  Global wall Y: {global_ceil_y:.0f} → {global_floor_y:.0f}")

    # ── G. PER-PLANE TILING ───────────────────────────────────
    result_img = room_img.astype(np.float64) / 255.0
    reg_top_dict = {}

    for label_id in np.unique(plane_labels):
        if label_id == 0:
            continue

        plane_mask_id = (plane_labels == label_id).astype(np.uint8)
        plane_clean   = np.clip(
            plane_mask_id.astype(np.int32) - obstacle_mask.astype(np.int32), 0, 1
        ).astype(np.uint8)

        n_visible = plane_clean.sum()
        n_total   = plane_mask_id.sum()
        print(f"  Plane {label_id}: total={n_total} visible={n_visible} px")

        if n_total < 50:
            print(f"    → too small, skipped")
            continue

        tiled, plane_reg_top, refined_mask = tile_plane_vp(
            room_img, plane_mask_id,
            depth, tile_unit,
            unit_w, unit_h, lighting, specular,
            global_ceil_y, global_floor_y,
            tile_img, tile_w_px, tile_h_px, grout_px)

        if tiled is None:
            print(f"    → geometry failed, attempting direct tiling")
            flat_canvas, _, _ = build_flat_tile_canvas(
                tile_img, tile_w_px, tile_h_px, grout_px,
                max(int(w/unit_w)+2,1), max(int(h/unit_h)+2,1))
            tiled = flat_canvas[:h, :w].astype(np.float64) / 255.0

            plane_ys, plane_xs = np.where(plane_mask_id > 0)
            if len(plane_ys) > 0:
                spec = specular[plane_ys, plane_xs, 0]
                for ch_i in range(3):
                    lc = lighting[plane_ys, plane_xs, ch_i]
                    tiled[plane_ys, plane_xs, ch_i] = np.clip(
                        tiled[plane_ys, plane_xs, ch_i] * (lc * 0.78 + 0.22) + spec * 0.25, 0, 1)

            plane_reg_top = None
            refined_mask = plane_mask_id # Fallback

        if plane_reg_top is not None:
            reg_top_dict[label_id] = plane_reg_top

        # Use the REFINED MASK for blending to get sharp straight edges
        # Obstacles are removed via subtraction from refined_mask
        clean_refined = np.clip(
            refined_mask.astype(np.int32) - obstacle_mask.astype(np.int32), 0, 1
        ).astype(np.uint8)

        # Soften edges slightly for anti-aliasing
        alpha = cv2.GaussianBlur(clean_refined.astype(np.float64), (3,3), 0)[:, :, np.newaxis]
        result_img = result_img * (1.0 - alpha) + tiled * alpha
        print(f"    → tiled OK")

    # ── H. RESTORE OBSTACLES ──────────────────────────────────
    obs_soft   = cv2.GaussianBlur(obstacle_mask.astype(np.float32), (11, 11), 0)[:, :, np.newaxis]
    orig_f     = room_img.astype(np.float64) / 255.0
    result_img = result_img * (1.0 - obs_soft) + orig_f * obs_soft

    final = (np.clip(result_img, 0, 1) * 255).astype(np.uint8)

    # ── I. PLANE VIZ ──────────────────────────────────────────
    colors    = [(220,50,50),(50,130,220),(50,200,100),(200,150,50),(150,50,200)]
    plane_viz = np.zeros((h, w, 3), dtype=np.uint8)
    for i, lid in enumerate(np.unique(plane_labels)):
        if lid == 0:
            continue
        plane_viz[plane_labels == lid] = colors[(i-1) % len(colors)]

    return final, plane_viz


# ─────────────────────────────────────────────────────────────
# RUNNER
# ─────────────────────────────────────────────────────────────
try:
    from google.colab import files

    print("Upload Room Image...")
    r = files.upload()
    print("Upload Tile Image...")
    t = files.upload()

    if r and t:
        room_cv = cv2.imdecode(np.frombuffer(list(r.values())[0], np.uint8), cv2.IMREAD_COLOR)
        tile_cv = cv2.imdecode(np.frombuffer(list(t.values())[0], np.uint8), cv2.IMREAD_COLOR)

        if room_cv is None or tile_cv is None:
            print("Error: Could not decode images.")
        else:
            u_width  = float(input("Tile Width  (cm) [60]: ") or 60)
            u_height = float(input("Tile Height (cm) [30]: ") or 30)
            u_orient = input("Orientation (h/v) [h]: ").strip().lower() or 'h'
            u_grout  = float(input("Grout thickness (mm) [3]: ") or 3)

            print("\nProcessing...")
            final_img, plane_viz = process_room_tiling(
                room_cv, tile_cv, u_width, u_height, u_orient, u_grout)

            fig, axes = plt.subplots(1, 3, figsize=(24, 8))
            axes[0].imshow(cv2.cvtColor(room_cv,   cv2.COLOR_BGR2RGB))
            axes[0].set_title("Original Room", fontsize=14); axes[0].axis('off')
            axes[1].imshow(plane_viz)
            axes[1].set_title("Wall Planes",   fontsize=14); axes[1].axis('off')
            axes[2].imshow(cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB))
            axes[2].set_title("Tiled Result",  fontsize=14); axes[2].axis('off')
            plt.tight_layout()
            plt.show()

            cv2.imwrite("tiled_result.jpg", final_img)

except ImportError:
    print("Not in Colab — set room_cv and tile_cv manually.")